# NAML PyTorch Implementation

This notebook migrates the Neural News Recommendation with Attentive Multi-View Learning (NAML) model from Keras/TensorFlow to PyTorch 2.2+ with CUDA 12.x support. The code keeps the original data processing pipeline and mirrors the model architecture using idiomatic PyTorch modules.


## Environment setup

The following cells define the imports, constants, and helper functions required for preprocessing data, constructing the PyTorch datasets, and implementing the NAML architecture. Each function now includes a descriptive docstring and inline comments to clarify its behavior.


In [ ]:
# Import statements for data handling, neural modeling, and evaluation utilities.
import itertools
import os
import random
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

import nltk
from nltk.tokenize import word_tokenize

# Download the Punkt tokenizer silently to ensure tokenization works out of the box.
nltk.download('punkt', quiet=True)


In [ ]:

# Define hyper-parameters that mirror the original TensorFlow notebook.
MAX_TITLE_LENGTH = 30  # Maximum number of tokens kept from each news title.
MAX_BODY_LENGTH = 300  # Maximum number of tokens kept from each news body.
MAX_HISTORY_LENGTH = 50  # Maximum length of the browsing history per user.
NEGATIVE_SAMPLES = 4  # Number of negative samples paired with each positive example during training.
EMBEDDING_DIM = 300  # Dimensionality of the pre-trained GloVe embeddings.

# Select the computation device (GPU when available, CPU otherwise).
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:

def set_random_seed(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch RNGs to obtain reproducible experiments.

    Args:
        seed: Integer seed used for every RNG in the pipeline.
    """
    random.seed(seed)  # Seed Python's built-in RNG.
    np.random.seed(seed)  # Seed NumPy's RNG so array operations are deterministic.
    torch.manual_seed(seed)  # Seed CPU side of PyTorch.
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)  # Seed every visible CUDA device.


def newsample(candidates: Sequence[str], ratio: int) -> List[str]:
    """Sample negative news identifiers with replacement awareness.

    The original notebook repeats the candidate list when the requested ratio
    exceeds its length; this helper preserves that behavior.

    Args:
        candidates: A sequence containing string identifiers of negative samples.
        ratio: Number of negatives required.

    Returns:
        A list of sampled candidate identifiers.
    """
    candidates = list(candidates)  # Ensure we can multiply the sequence if needed.
    if ratio > len(candidates) and len(candidates) > 0:
        # Repeat the candidate list enough times to support sampling without bias.
        expanded = candidates * (ratio // len(candidates) + 1)
        return random.sample(expanded, ratio)
    # Otherwise draw directly from the available candidates.
    return random.sample(candidates, ratio) if len(candidates) > 0 else []


def pad_or_truncate(sequence: Sequence[int], max_length: int, padding_value: int = 0) -> List[int]:
    """Pad or truncate a sequence to an exact length.

    Args:
        sequence: Iterable of integers representing token identifiers.
        max_length: Desired output length after padding/truncation.
        padding_value: Integer used when additional padding positions are required.

    Returns:
        A list with exactly `max_length` integers.
    """
    seq = list(sequence)[:max_length]  # Trim tokens that exceed the maximum length.
    # Extend the sequence with the padding value until it reaches the target size.
    seq += [padding_value] * (max_length - len(seq))
    return seq


In [ ]:

def preprocess_news_file(
    file_path: str,
    max_title_length: int = MAX_TITLE_LENGTH,
    max_body_length: int = MAX_BODY_LENGTH,
    min_word_freq: int = 3,
) -> Tuple[Dict[str, Tuple[int, int]], Dict[str, int], Dict[str, int], np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, int]]:
    """Parse the news metadata file and build tokenized tensors for every news article.

    Args:
        file_path: Location of the TSV file describing news attributes.
        max_title_length: Maximum number of title tokens preserved per article.
        max_body_length: Maximum number of body tokens preserved per article.
        min_word_freq: Minimum token frequency required to keep a word in the vocabulary.

    Returns:
        A tuple containing:
            * word_dict: Mapping from token string to `(index, frequency)` pairs.
            * category_dict: Mapping from category name to integer index.
            * subcategory_dict: Mapping from sub-category name to integer index.
            * news_titles: Array of token ids for every news title (index 0 is padding).
            * news_bodies: Array of token ids for every news body (index 0 is padding).
            * news_categories: Array of category ids for every news item.
            * news_subcategories: Array of sub-category ids for every news item.
            * news_index: Dictionary mapping external news identifiers to row indices.
    """
    news: Dict[str, Tuple[str, str, List[str], List[str]]] = {}  # Stores raw parsed information.
    category_dict: Dict[str, int] = {'None': 0}  # Category vocabulary with padding entry.
    subcategory_dict: Dict[str, int] = {'None': 0}  # Sub-category vocabulary with padding entry.

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('	')  # Split TSV columns.
            if len(parts) < 8:
                continue  # Skip malformed lines.
            news_id = parts[1]
            category = parts[2] or 'None'
            subcategory = parts[3] or 'None'
            # Tokenize title and body using NLTK's Punkt tokenizer.
            title_tokens = word_tokenize(parts[6].lower())
            body_tokens = word_tokenize(parts[7].lower())
            news[news_id] = (category, subcategory, title_tokens, body_tokens)
            if category not in category_dict:
                category_dict[category] = len(category_dict)
            if subcategory not in subcategory_dict:
                subcategory_dict[subcategory] = len(subcategory_dict)

    # Build a raw word dictionary that counts every token occurrence.
    word_dict_raw: Dict[str, Tuple[int, int]] = {'PADDING': (0, 10**9)}
    for _, _, title_tokens, body_tokens in news.values():
        for token in title_tokens + body_tokens:
            if token in word_dict_raw:
                # Update the existing count for known tokens.
                index, count = word_dict_raw[token]
                word_dict_raw[token] = (index, count + 1)
            else:
                # Assign a fresh index and initialize the count to one.
                word_dict_raw[token] = (len(word_dict_raw), 1)

    # Prune infrequent words to limit vocabulary size.
    word_dict: Dict[str, Tuple[int, int]] = {}
    for token, (index, count) in word_dict_raw.items():
        if count >= min_word_freq or token == 'PADDING':
            word_dict[token] = (len(word_dict), count)

    # Initialize containers with a dedicated zero row for padding.
    news_titles: List[List[int]] = [pad_or_truncate([], max_title_length)]
    news_bodies: List[List[int]] = [pad_or_truncate([], max_body_length)]
    news_categories: List[List[int]] = [[0]]
    news_subcategories: List[List[int]] = [[0]]
    news_index: Dict[str, int] = {'0': 0}

    for news_id, (category, subcategory, title_tokens, body_tokens) in news.items():
        title_ids = [word_dict[token][0] for token in title_tokens if token in word_dict]
        body_ids = [word_dict[token][0] for token in body_tokens if token in word_dict]
        padded_title = pad_or_truncate(title_ids, max_title_length)
        padded_body = pad_or_truncate(body_ids, max_body_length)
        news_index[news_id] = len(news_titles)
        news_titles.append(padded_title)
        news_bodies.append(padded_body)
        news_categories.append([category_dict.get(category, 0)])
        news_subcategories.append([subcategory_dict.get(subcategory, 0)])

    # Convert lists into NumPy arrays that can later be indexed efficiently.
    news_titles_array = np.asarray(news_titles, dtype='int32')
    news_bodies_array = np.asarray(news_bodies, dtype='int32')
    news_categories_array = np.asarray(news_categories, dtype='int32')
    news_subcategories_array = np.asarray(news_subcategories, dtype='int32')

    return (
        word_dict,
        category_dict,
        subcategory_dict,
        news_titles_array,
        news_bodies_array,
        news_categories_array,
        news_subcategories_array,
        news_index,
    )


def preprocess_user_file(
    file_path: str,
    npratio: int = NEGATIVE_SAMPLES,
    history_size: int = MAX_HISTORY_LENGTH,
) -> Tuple[Dict[str, int], np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[Tuple[int, int]]]:
    """Parse user interaction logs and build training/evaluation matrices.

    Args:
        file_path: Location of the TSV log containing impressions per user.
        npratio: Number of negative samples paired with each positive impression.
        history_size: Maximum number of clicked items stored as browsing history.

    Returns:
        A tuple mirroring the tensors used by the original notebook:
            * user_dict: Mapping from user ids to contiguous indices.
            * train_candidates: Array of candidate news indices for training (num_samples, npratio+1).
            * train_labels: Array of one-hot labels aligned with train_candidates.
            * train_user_indices: Array linking each training sample to a user index.
            * test_candidates: Array of news indices for evaluation (flattened per candidate).
            * test_labels: Binary labels corresponding to test_candidates.
            * test_user_indices: Array linking evaluation rows to user indices.
            * train_histories: Browsing history indices for every training sample.
            * test_histories: Browsing history indices for every evaluation row.
            * test_sessions: Start/end index pairs describing impression boundaries.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        user_lines = f.readlines()

    user_dict: Dict[str, int] = {}
    for raw_line in user_lines:
        user_id = raw_line.strip().split('	')[0]
        if user_id not in user_dict:
            user_dict[user_id] = len(user_dict)

    train_candidates: List[List[int]] = []
    train_labels: List[List[int]] = []
    train_user_indices: List[int] = []
    test_candidates: List[int] = []
    test_labels: List[int] = []
    test_user_indices: List[int] = []
    train_histories: List[List[int]] = []
    test_histories: List[List[int]] = []
    test_sessions: List[Tuple[int, int]] = []

    for raw_line in user_lines:
        line = raw_line.strip().split('	')
        user_id = line[0]
        user_index = user_dict[user_id]
        if len(line) < 3:
            continue  # Skip rows without impression information.
        impressions = [segment.split('#TAB#') for segment in line[2].split('#N#') if segment]
        train_pos = [segment[0].split() if len(segment) > 0 else [] for segment in impressions]
        train_neg = [segment[1].split() if len(segment) > 1 else [] for segment in impressions]
        positive_pool = list(itertools.chain.from_iterable(train_pos)) if train_pos else []

        # Construct evaluation impressions when available.
        if len(line) >= 4:
            test_impressions = [segment.split('#TAB#') for segment in line[3].split('#N#') if segment]
            test_pos = [segment[0].split() if len(segment) > 0 else [] for segment in test_impressions]
            test_neg = [segment[1].split() if len(segment) > 1 else [] for segment in test_impressions]
            for pos_list, neg_list in zip(test_pos, test_neg):
                session_start = len(test_candidates)
                # Sample the browsing history for evaluation users (capped to `history_size`).
                history_candidates = list(set(positive_pool))
                sampled_history = [int(x) for x in random.sample(history_candidates, min(history_size, len(history_candidates)))[:history_size]]
                sampled_history += [0] * (history_size - len(sampled_history))
                for candidate in pos_list:
                    test_candidates.append(int(candidate))
                    test_labels.append(1)
                    test_user_indices.append(user_index)
                    test_histories.append(sampled_history)
                for candidate in neg_list:
                    test_candidates.append(int(candidate))
                    test_labels.append(0)
                    test_user_indices.append(user_index)
                    test_histories.append(sampled_history)
                session_end = len(test_candidates)
                test_sessions.append((session_start, session_end))

        # Build training triples using negative sampling.
        for impression_pos, impression_neg in zip(train_pos, train_neg):
            for positive_candidate in impression_pos:
                negatives = newsample(impression_neg, npratio)
                candidate_group = negatives + [positive_candidate]
                labels = [0] * len(negatives) + [1]
                permutation = list(range(len(candidate_group)))
                random.shuffle(permutation)
                shuffled_candidates = [int(candidate_group[idx]) for idx in permutation]
                shuffled_labels = [labels[idx] for idx in permutation]
                history_candidates = list(set(positive_pool) - {positive_candidate})
                sampled_history = [int(x) for x in random.sample(history_candidates, min(history_size, len(history_candidates)))[:history_size]]
                sampled_history += [0] * (history_size - len(sampled_history))
                train_candidates.append(shuffled_candidates)
                train_labels.append(shuffled_labels)
                train_user_indices.append(user_index)
                train_histories.append(sampled_history)

    # Convert lists into NumPy arrays that will later be transformed into PyTorch tensors.
    train_candidates_array = np.asarray(train_candidates, dtype='int32')
    train_labels_array = np.asarray(train_labels, dtype='int32')
    train_user_indices_array = np.asarray(train_user_indices, dtype='int32')
    test_candidates_array = np.asarray(test_candidates, dtype='int32')
    test_labels_array = np.asarray(test_labels, dtype='int32')
    test_user_indices_array = np.asarray(test_user_indices, dtype='int32')
    train_histories_array = np.asarray(train_histories, dtype='int32')
    test_histories_array = np.asarray(test_histories, dtype='int32')

    return (
        user_dict,
        train_candidates_array,
        train_labels_array,
        train_user_indices_array,
        test_candidates_array,
        test_labels_array,
        test_user_indices_array,
        train_histories_array,
        test_histories_array,
        test_sessions,
    )


def build_embedding_matrix(
    word_dict: Dict[str, Tuple[int, int]],
    embedding_path: str,
    embedding_dim: int = EMBEDDING_DIM,
) -> np.ndarray:
    """Create an embedding matrix initialized from pre-trained GloVe vectors.

    Args:
        word_dict: Vocabulary mapping used to look up embedding rows.
        embedding_path: File path pointing to a GloVe text file.
        embedding_dim: Dimensionality of the vectors to load.

    Returns:
        A NumPy array shaped `(vocab_size, embedding_dim)` suitable for `nn.Embedding`.
    """
    vocab_size = len(word_dict)
    embedding_matrix = np.random.normal(0, 0.1, size=(vocab_size, embedding_dim)).astype('float32')
    embedding_matrix[0] = 0.0  # Ensure the padding row stays zero-initialized.

    if not embedding_path or not os.path.exists(embedding_path):
        # Fall back to random initialization when the GloVe file is unavailable.
        print(f"Embedding file {embedding_path!r} not found. Using random initialization.")
        return embedding_matrix

    found = 0
    with open(embedding_path, 'rb') as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            token = parts[0].decode('utf-8', errors='ignore')
            if token in word_dict:
                vector = np.asarray(parts[1:], dtype='float32')
                if vector.shape[0] != embedding_dim:
                    continue
                index = word_dict[token][0]
                embedding_matrix[index] = vector
                found += 1
    print(f"Loaded {found} pre-trained embeddings out of {vocab_size} tokens.")
    return embedding_matrix


In [ ]:
class NAMLTrainingDataset(Dataset):
    """Dataset returning candidate groups and user histories for training.

    The dataset mimics the shape produced by the original generator so the model
    can compute a softmax over `npratio + 1` candidates per impression.
    """

    def __init__(
        self,
        candidate_indices: np.ndarray,
        labels: np.ndarray,
        user_indices: np.ndarray,
        histories: np.ndarray,
        news_titles: np.ndarray,
        news_bodies: np.ndarray,
        news_categories: np.ndarray,
        news_subcategories: np.ndarray,
    ) -> None:
        """Store references to the preprocessed tensors.

        Args:
            candidate_indices: Array with shape (num_samples, n_candidates).
            labels: One-hot labels aligned with `candidate_indices`.
            user_indices: Identifier mapping each row to a user.
            histories: Array of browsing history news indices per row.
            news_titles/news_bodies/news_categories/news_subcategories: Feature tables for news ids.
        """
        super().__init__()
        self.candidate_indices = candidate_indices
        self.labels = labels
        self.user_indices = user_indices
        self.histories = histories
        self.news_titles = news_titles
        self.news_bodies = news_bodies
        self.news_categories = news_categories
        self.news_subcategories = news_subcategories

    def __len__(self) -> int:
        """Return the number of training impressions available."""
        return len(self.candidate_indices)

    def __getitem__(self, idx: int) -> Dict[str, np.ndarray]:
        """Fetch the candidate features and user history for one impression.

        Args:
            idx: Row index describing the impression to retrieve.

        Returns:
            A dictionary containing all the tensors required by the model.
        """
        candidate_ids = self.candidate_indices[idx]
        history_ids = self.histories[idx]
        sample = {
            'candidate_titles': self.news_titles[candidate_ids],
            'candidate_bodies': self.news_bodies[candidate_ids],
            'candidate_categories': self.news_categories[candidate_ids],
            'candidate_subcategories': self.news_subcategories[candidate_ids],
            'history_titles': self.news_titles[history_ids],
            'history_bodies': self.news_bodies[history_ids],
            'history_categories': self.news_categories[history_ids],
            'history_subcategories': self.news_subcategories[history_ids],
            'labels': self.labels[idx],
            'user_indices': self.user_indices[idx],
        }
        return sample


class NAMLEvaluationDataset(Dataset):
    """Dataset returning individual candidates for evaluation.

    Each row represents a single news item paired with the corresponding user
    history so that the model can score candidates independently.
    """

    def __init__(
        self,
        candidate_indices: np.ndarray,
        labels: np.ndarray,
        user_indices: np.ndarray,
        histories: np.ndarray,
        news_titles: np.ndarray,
        news_bodies: np.ndarray,
        news_categories: np.ndarray,
        news_subcategories: np.ndarray,
        session_boundaries: List[Tuple[int, int]],
    ) -> None:
        """Initialize the evaluation dataset with flattened candidate rows."""
        super().__init__()
        self.candidate_indices = candidate_indices
        self.labels = labels
        self.user_indices = user_indices
        self.histories = histories
        self.news_titles = news_titles
        self.news_bodies = news_bodies
        self.news_categories = news_categories
        self.news_subcategories = news_subcategories
        self.session_boundaries = session_boundaries
        # Pre-compute a session id per row to simplify metric aggregation later on.
        self.session_ids = np.zeros(len(candidate_indices), dtype='int32')
        for session_id, (start, end) in enumerate(session_boundaries):
            self.session_ids[start:end] = session_id

    def __len__(self) -> int:
        """Return the total number of evaluation rows."""
        return len(self.candidate_indices)

    def __getitem__(self, idx: int) -> Dict[str, np.ndarray]:
        """Gather all features needed to score a single candidate.

        Args:
            idx: Row index of the candidate to load.

        Returns:
            A dictionary with candidate/user features and bookkeeping metadata.
        """
        candidate_id = self.candidate_indices[idx]
        history_ids = self.histories[idx]
        sample = {
            'candidate_titles': self.news_titles[[candidate_id]],
            'candidate_bodies': self.news_bodies[[candidate_id]],
            'candidate_categories': self.news_categories[[candidate_id]],
            'candidate_subcategories': self.news_subcategories[[candidate_id]],
            'history_titles': self.news_titles[history_ids],
            'history_bodies': self.news_bodies[history_ids],
            'history_categories': self.news_categories[history_ids],
            'history_subcategories': self.news_subcategories[history_ids],
            'labels': np.asarray([self.labels[idx]], dtype='int32'),
            'user_indices': self.user_indices[idx],
            'session_id': self.session_ids[idx],
        }
        return sample


def collate_train_batch(batch: List[Dict[str, np.ndarray]]) -> Dict[str, torch.Tensor]:
    """Stack a list of training samples into batched PyTorch tensors."""
    candidate_titles = torch.as_tensor(np.stack([item['candidate_titles'] for item in batch]), dtype=torch.long)
    candidate_bodies = torch.as_tensor(np.stack([item['candidate_bodies'] for item in batch]), dtype=torch.long)
    candidate_categories = torch.as_tensor(np.stack([item['candidate_categories'] for item in batch]), dtype=torch.long)
    candidate_subcategories = torch.as_tensor(np.stack([item['candidate_subcategories'] for item in batch]), dtype=torch.long)
    history_titles = torch.as_tensor(np.stack([item['history_titles'] for item in batch]), dtype=torch.long)
    history_bodies = torch.as_tensor(np.stack([item['history_bodies'] for item in batch]), dtype=torch.long)
    history_categories = torch.as_tensor(np.stack([item['history_categories'] for item in batch]), dtype=torch.long)
    history_subcategories = torch.as_tensor(np.stack([item['history_subcategories'] for item in batch]), dtype=torch.long)
    label_vectors = torch.as_tensor(np.stack([item['labels'] for item in batch]), dtype=torch.float32)
    label_indices = torch.argmax(label_vectors, dim=1)
    return {
        'candidate_titles': candidate_titles,
        'candidate_bodies': candidate_bodies,
        'candidate_categories': candidate_categories,
        'candidate_subcategories': candidate_subcategories,
        'history_titles': history_titles,
        'history_bodies': history_bodies,
        'history_categories': history_categories,
        'history_subcategories': history_subcategories,
        'label_indices': label_indices,
        'label_vectors': label_vectors,
    }


def collate_eval_batch(batch: List[Dict[str, np.ndarray]]) -> Dict[str, torch.Tensor]:
    """Stack evaluation samples while preserving session metadata."""
    candidate_titles = torch.as_tensor(np.stack([item['candidate_titles'] for item in batch]), dtype=torch.long)
    candidate_bodies = torch.as_tensor(np.stack([item['candidate_bodies'] for item in batch]), dtype=torch.long)
    candidate_categories = torch.as_tensor(np.stack([item['candidate_categories'] for item in batch]), dtype=torch.long)
    candidate_subcategories = torch.as_tensor(np.stack([item['candidate_subcategories'] for item in batch]), dtype=torch.long)
    history_titles = torch.as_tensor(np.stack([item['history_titles'] for item in batch]), dtype=torch.long)
    history_bodies = torch.as_tensor(np.stack([item['history_bodies'] for item in batch]), dtype=torch.long)
    history_categories = torch.as_tensor(np.stack([item['history_categories'] for item in batch]), dtype=torch.long)
    history_subcategories = torch.as_tensor(np.stack([item['history_subcategories'] for item in batch]), dtype=torch.long)
    labels = torch.as_tensor(np.concatenate([item['labels'] for item in batch]), dtype=torch.float32)
    session_ids = torch.as_tensor([item['session_id'] for item in batch], dtype=torch.long)
    return {
        'candidate_titles': candidate_titles,
        'candidate_bodies': candidate_bodies,
        'candidate_categories': candidate_categories,
        'candidate_subcategories': candidate_subcategories,
        'history_titles': history_titles,
        'history_bodies': history_bodies,
        'history_categories': history_categories,
        'history_subcategories': history_subcategories,
        'labels': labels,
        'session_ids': session_ids,
    }


In [ ]:
class AdditiveAttention(nn.Module):
    """Classic additive attention module used for token and view selection."""
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        """Initialize the additive attention projection layers."""
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
    def forward(self, inputs: torch.Tensor, mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """Compute attention weights over the sequence dimension.
        Args:
            inputs: Tensor shaped `(batch, sequence_length, input_dim)`.
            mask: Optional boolean tensor marking valid positions (True = keep).
        Returns:
            Tuple containing the weighted sum vector and the attention weights.
        """
        scores = self.projection(inputs).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        weighted = torch.sum(inputs * weights.unsqueeze(-1), dim=1)
        return weighted, weights
class NewsEncoder(nn.Module):
    """Encode multi-view news features into a single dense vector."""
    def __init__(
        self,
        embedding_matrix: np.ndarray,
        category_vocab_size: int,
        subcategory_vocab_size: int,
        embedding_dim: int = EMBEDDING_DIM,
        cnn_filters: int = 400,
        attention_hidden: int = 200,
        category_dim: int = 50,
        dropout: float = 0.2,
    ) -> None:
        """Configure the sub-modules that encode news content and metadata."""
        super().__init__()
        # Shared word embedding layer initialized from the pre-computed matrix.
        self.word_embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False,
            padding_idx=0,
        )
        self.dropout = nn.Dropout(dropout)
        # Title encoder using a temporal CNN with kernel size 3.
        self.title_cnn = nn.Conv1d(embedding_dim, cnn_filters, kernel_size=3, padding=1)
        self.title_attention = AdditiveAttention(cnn_filters, attention_hidden)
        # Body encoder reuses the same configuration as the title encoder.
        self.body_cnn = nn.Conv1d(embedding_dim, cnn_filters, kernel_size=3, padding=1)
        self.body_attention = AdditiveAttention(cnn_filters, attention_hidden)
        # Category and sub-category embeddings followed by dense projections.
        self.category_embedding = nn.Embedding(category_vocab_size, category_dim, padding_idx=0)
        self.subcategory_embedding = nn.Embedding(subcategory_vocab_size, category_dim, padding_idx=0)
        self.category_projection = nn.Sequential(nn.Flatten(), nn.Linear(category_dim, cnn_filters), nn.ReLU())
        self.subcategory_projection = nn.Sequential(nn.Flatten(), nn.Linear(category_dim, cnn_filters), nn.ReLU())
        # View-level attention fuses the four representations.
        self.view_attention = AdditiveAttention(cnn_filters, attention_hidden)
    def forward(
        self,
        title_inputs: torch.Tensor,
        body_inputs: torch.Tensor,
        category_inputs: torch.Tensor,
        subcategory_inputs: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """Encode raw news features into a dense representation.
        Args:
            title_inputs: Token ids for titles `(batch, seq_len)`.
            body_inputs: Token ids for bodies `(batch, seq_len)`.
            category_inputs: Category ids `(batch, 1)`.
            subcategory_inputs: Sub-category ids `(batch, 1)`.
        Returns:
            Encoded news vector along with intermediate attention weights.
        """
        title_emb = self.dropout(self.word_embedding(title_inputs)).transpose(1, 2)
        title_features = torch.relu(self.title_cnn(title_emb)).transpose(1, 2)
        title_features = self.dropout(title_features)
        title_mask = title_inputs != 0
        title_vector, title_weights = self.title_attention(title_features, title_mask)
        body_emb = self.dropout(self.word_embedding(body_inputs)).transpose(1, 2)
        body_features = torch.relu(self.body_cnn(body_emb)).transpose(1, 2)
        body_features = self.dropout(body_features)
        body_mask = body_inputs != 0
        body_vector, body_weights = self.body_attention(body_features, body_mask)
        category_vector = self.category_projection(self.category_embedding(category_inputs))
        subcategory_vector = self.subcategory_projection(self.subcategory_embedding(subcategory_inputs))
        stacked_views = torch.stack(
            [title_vector, body_vector, category_vector, subcategory_vector], dim=1
        )
        news_vector, view_weights = self.view_attention(stacked_views)
        attention_info = {
            'title_weights': title_weights,
            'body_weights': body_weights,
            'view_weights': view_weights,
        }
        return news_vector, attention_info
class UserEncoder(nn.Module):
    """Aggregate a sequence of browsed news vectors into a user representation."""
    def __init__(self, input_dim: int, attention_hidden: int = 200, dropout: float = 0.2) -> None:
        """Store the attention layer used to summarize user histories."""
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.attention = AdditiveAttention(input_dim, attention_hidden)
    def forward(self, history_vectors: torch.Tensor, history_mask: torch.Tensor | None = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode the browsing history into a single dense vector.
        Args:
            history_vectors: Tensor `(batch, history_length, input_dim)`.
            history_mask: Optional boolean mask for padded positions.
        Returns:
            Tuple with the user vector and attention weights over the history.
        """
        history_vectors = self.dropout(history_vectors)
        user_vector, history_weights = self.attention(history_vectors, history_mask)
        return user_vector, history_weights
class NAML(nn.Module):
    """Full NAML recommendation model combining news and user encoders."""
    def __init__(self, news_encoder: NewsEncoder, user_encoder: UserEncoder) -> None:
        """Attach the news and user encoder components to the model."""
        super().__init__()
        self.news_encoder = news_encoder
        self.user_encoder = user_encoder
    def encode_news_batch(
        self,
        titles: torch.Tensor,
        bodies: torch.Tensor,
        categories: torch.Tensor,
        subcategories: torch.Tensor,
    ) -> torch.Tensor:
        """Encode a batch of news items into dense vectors."""
        news_vectors, _ = self.news_encoder(titles, bodies, categories, subcategories)
        return news_vectors
    def forward(
        self,
        candidate_titles: torch.Tensor,
        candidate_bodies: torch.Tensor,
        candidate_categories: torch.Tensor,
        candidate_subcategories: torch.Tensor,
        history_titles: torch.Tensor,
        history_bodies: torch.Tensor,
        history_categories: torch.Tensor,
        history_subcategories: torch.Tensor,
    ) -> torch.Tensor:
        """Compute logits over candidate news given the user history.
        Args:
            candidate_*: Batched candidate features `(batch, num_candidates, feature_dim)`.
            history_*: Batched history features `(batch, history_length, feature_dim)`.
        Returns:
            Logits shaped `(batch, num_candidates)` ready for cross-entropy loss.
        """
        batch_size, num_candidates, _ = candidate_titles.shape
        history_length = history_titles.shape[1]
        # Flatten candidates to encode them with the shared news encoder.
        flattened_titles = candidate_titles.view(batch_size * num_candidates, -1)
        flattened_bodies = candidate_bodies.view(batch_size * num_candidates, -1)
        flattened_categories = candidate_categories.view(batch_size * num_candidates, -1)
        flattened_subcategories = candidate_subcategories.view(batch_size * num_candidates, -1)
        candidate_vectors, _ = self.news_encoder(
            flattened_titles,
            flattened_bodies,
            flattened_categories,
            flattened_subcategories,
        )
        candidate_vectors = candidate_vectors.view(batch_size, num_candidates, -1)
        # Encode the browsing history in the same fashion.
        history_titles_flat = history_titles.view(batch_size * history_length, -1)
        history_bodies_flat = history_bodies.view(batch_size * history_length, -1)
        history_categories_flat = history_categories.view(batch_size * history_length, -1)
        history_subcategories_flat = history_subcategories.view(batch_size * history_length, -1)
        history_vectors, _ = self.news_encoder(
            history_titles_flat,
            history_bodies_flat,
            history_categories_flat,
            history_subcategories_flat,
        )
        history_vectors = history_vectors.view(batch_size, history_length, -1)
        history_mask = (history_titles.sum(dim=-1) != 0)
        user_vector, _ = self.user_encoder(history_vectors, history_mask)
        # Compute dot-product scores between the user vector and each candidate.
        logits = torch.bmm(candidate_vectors, user_vector.unsqueeze(-1)).squeeze(-1)
        return logits
    def score_single_candidate(
        self,
        candidate_titles: torch.Tensor,
        candidate_bodies: torch.Tensor,
        candidate_categories: torch.Tensor,
        candidate_subcategories: torch.Tensor,
        history_titles: torch.Tensor,
        history_bodies: torch.Tensor,
        history_categories: torch.Tensor,
        history_subcategories: torch.Tensor,
    ) -> torch.Tensor:
        """Score individual candidates with sigmoid outputs for evaluation."""
        logits = self.forward(
            candidate_titles,
            candidate_bodies,
            candidate_categories,
            candidate_subcategories,
            history_titles,
            history_bodies,
            history_categories,
            history_subcategories,
        )
        return torch.sigmoid(logits)


In [ ]:

def build_naml_model(
    embedding_matrix: np.ndarray,
    news_categories: np.ndarray,
    news_subcategories: np.ndarray,
    device: torch.device = DEVICE,
) -> NAML:
    """Factory helper that instantiates the NAML architecture."""
    category_vocab_size = int(news_categories.max()) + 1 if news_categories.size > 0 else 1
    subcategory_vocab_size = int(news_subcategories.max()) + 1 if news_subcategories.size > 0 else 1
    news_encoder = NewsEncoder(
        embedding_matrix=embedding_matrix,
        category_vocab_size=category_vocab_size,
        subcategory_vocab_size=subcategory_vocab_size,
    )
    user_encoder = UserEncoder(input_dim=400)
    model = NAML(news_encoder, user_encoder)
    return model.to(device)


def train_one_epoch(
    model: NAML,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> float:
    """Run a single training epoch and report the average loss."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    for batch in dataloader:
        optimizer.zero_grad()
        logits = model(
            batch['candidate_titles'].to(device),
            batch['candidate_bodies'].to(device),
            batch['candidate_categories'].to(device),
            batch['candidate_subcategories'].to(device),
            batch['history_titles'].to(device),
            batch['history_bodies'].to(device),
            batch['history_categories'].to(device),
            batch['history_subcategories'].to(device),
        )
        labels = batch['label_indices'].to(device)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
    return total_loss / max(total_samples, 1)


def compute_dcg(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    """Compute Discounted Cumulative Gain at rank `k`."""
    order = np.argsort(y_score)[::-1][:k]
    gains = (2 ** y_true[order] - 1)
    discounts = np.log2(np.arange(order.size) + 2)
    return float(np.sum(gains / discounts))


def compute_ndcg(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    """Compute Normalized Discounted Cumulative Gain at rank `k`."""
    ideal = compute_dcg(y_true, y_true, k)
    if ideal == 0:
        return 0.0
    return compute_dcg(y_true, y_score, k) / ideal


def compute_mrr(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Compute Mean Reciprocal Rank for a single impression."""
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order]
    ranks = np.arange(1, y_true_sorted.size + 1)
    reciprocal = (y_true_sorted / ranks)
    positives = y_true_sorted.sum()
    return float(np.sum(reciprocal) / positives) if positives > 0 else 0.0


def compute_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Compute ROC-AUC using the rank statistic (without sklearn)."""
    y_true = y_true.astype(float)
    y_score = y_score.astype(float)
    pos = y_true.sum()
    neg = y_true.size - pos
    if pos == 0 or neg == 0:
        return float('nan')
    order = np.argsort(y_score)
    ranks = np.arange(1, y_true.size + 1)
    pos_ranks = ranks[order][y_true[order] == 1]
    auc = (pos_ranks.sum() - pos * (pos + 1) / 2) / (pos * neg)
    return float(auc)


def summarize_session_metrics(
    session_boundaries: List[Tuple[int, int]],
    labels: np.ndarray,
    scores: np.ndarray,
) -> Dict[str, float]:
    """Aggregate ranking metrics over all evaluation sessions."""
    auc_list: List[float] = []
    mrr_list: List[float] = []
    ndcg5_list: List[float] = []
    ndcg10_list: List[float] = []
    for start, end in session_boundaries:
        session_labels = labels[start:end]
        session_scores = scores[start:end]
        if session_labels.sum() == 0:
            continue
        auc_list.append(compute_roc_auc(session_labels, session_scores))
        mrr_list.append(compute_mrr(session_labels, session_scores))
        ndcg5_list.append(compute_ndcg(session_labels, session_scores, k=5))
        ndcg10_list.append(compute_ndcg(session_labels, session_scores, k=10))
    return {
        'AUC': float(np.nanmean(auc_list)) if auc_list else float('nan'),
        'MRR': float(np.mean(mrr_list)) if mrr_list else 0.0,
        'nDCG@5': float(np.mean(ndcg5_list)) if ndcg5_list else 0.0,
        'nDCG@10': float(np.mean(ndcg10_list)) if ndcg10_list else 0.0,
    }


def evaluate(
    model: NAML,
    dataloader: DataLoader,
    session_boundaries: List[Tuple[int, int]],
    device: torch.device = DEVICE,
) -> Dict[str, float]:
    """Evaluate the model on flattened candidate impressions."""
    model.eval()
    all_scores: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    with torch.no_grad():
        for batch in dataloader:
            scores = model.score_single_candidate(
                batch['candidate_titles'].to(device),
                batch['candidate_bodies'].to(device),
                batch['candidate_categories'].to(device),
                batch['candidate_subcategories'].to(device),
                batch['history_titles'].to(device),
                batch['history_bodies'].to(device),
                batch['history_categories'].to(device),
                batch['history_subcategories'].to(device),
            )
            all_scores.append(scores.squeeze(-1).cpu().numpy())
            all_labels.append(batch['labels'].cpu().numpy())
    concatenated_scores = np.concatenate(all_scores, axis=0)
    concatenated_labels = np.concatenate(all_labels, axis=0)
    return summarize_session_metrics(session_boundaries, concatenated_labels, concatenated_scores)


In [ ]:

# Example usage demonstrating the full PyTorch workflow on the sample data.
set_random_seed(42)

# Preprocess the news and user logs shipped with the repository.
(
    word_dict,
    category_dict,
    subcategory_dict,
    news_titles,
    news_bodies,
    news_categories,
    news_subcategories,
    news_index,
) = preprocess_news_file('DocMeta_sample.tsv')
(
    user_dict,
    train_candidates,
    train_labels,
    train_user_indices,
    test_candidates,
    test_labels,
    test_user_indices,
    train_histories,
    test_histories,
    test_sessions,
) = preprocess_user_file('ClickData_sample.tsv')

# Initialize embeddings (random by default when GloVe is not present).
embedding_matrix = build_embedding_matrix(word_dict, 'glove.840B.300d.txt')
model = build_naml_model(embedding_matrix, news_categories, news_subcategories, device=DEVICE)

# Construct PyTorch datasets and loaders.
train_dataset = NAMLTrainingDataset(
    candidate_indices=train_candidates,
    labels=train_labels,
    user_indices=train_user_indices,
    histories=train_histories,
    news_titles=news_titles,
    news_bodies=news_bodies,
    news_categories=news_categories,
    news_subcategories=news_subcategories,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_train_batch,
)

eval_dataset = NAMLEvaluationDataset(
    candidate_indices=test_candidates,
    labels=test_labels,
    user_indices=test_user_indices,
    histories=test_histories,
    news_titles=news_titles,
    news_bodies=news_bodies,
    news_categories=news_categories,
    news_subcategories=news_subcategories,
    session_boundaries=test_sessions,
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_eval_batch,
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Run a tiny demonstration training loop (epochs can be increased as needed).
for epoch in range(1):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, device=DEVICE)
    metrics = evaluate(model, eval_loader, test_sessions, device=DEVICE)
    print(f'Epoch {epoch + 1} - Loss: {loss:.4f} | Metrics: {metrics}')
